# Aequitas — Data Understanding Notebook

**Purpose:** Build deep understanding of every dataset before writing pipeline code.  
**Approach:** Start with individual LSOAs, zoom out to distributions, then map everything to the 8 socio-economic factors.

This is NOT an audit. The audit (`01_data_audit.ipynb`) confirmed the data is clean.  
This notebook asks: **what does the data actually tell us?**

In [ ]:
import sys
import json
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", font_scale=0.9)

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"

# ── Load ALL datasets (England only) ──────────────────────────────────────
naptan_raw = pd.read_csv(RAW / "naptan" / "Stops.csv", low_memory=False)
stops = naptan_raw[
    naptan_raw["StopType"].isin({"BCT", "BCS", "BCE"})
    & (naptan_raw["Status"] == "active")
    & naptan_raw["ATCOCode"].str.match(r"^[01234]", na=False)
    & naptan_raw["Latitude"].notna()
].copy()

imd = pd.read_csv(RAW / "imd" / "imd2025_all_ranks_scores_deciles.csv")

census = pd.read_csv(RAW / "census" / "census2021_ts001_lsoa_population.csv")
census = census[census["geography code"].str.startswith("E", na=False)].copy()

nomis = pd.read_csv(RAW / "nomis" / "nomis_unemployment_lsoa.csv")
nomis = nomis[nomis["geography code"].str.startswith("E", na=False)].copy()

ruc = pd.read_csv(RAW / "census" / "ruc2021_lsoa_ew.csv")
ruc = ruc[ruc["LSOA21CD"].str.startswith("E", na=False)].copy()

with zipfile.ZipFile(RAW / "census" / "census2021-ts007a.zip") as zf:
    age = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
age = age[age["geography code"].str.startswith("E", na=False)].copy()

with zipfile.ZipFile(RAW / "census" / "census2021-ts045.zip") as zf:
    cars = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
cars = cars[cars["geography code"].str.startswith("E", na=False)].copy()

with zipfile.ZipFile(RAW / "census" / "census2021-ts021.zip") as zf:
    eth = pd.read_csv(zf.open(next(n for n in zf.namelist() if "lsoa" in n.lower() and n.endswith(".csv"))))
eth = eth[eth["geography code"].str.startswith("E", na=False)].copy()

print(f"All datasets loaded.")
print(f"  Stops:     {len(stops):,}")
print(f"  IMD:       {len(imd):,}")
print(f"  Census:    {len(census):,}")
print(f"  NOMIS:     {len(nomis):,}")
print(f"  RUC:       {len(ruc):,}")
print(f"  Age:       {len(age):,}")
print(f"  Cars:      {len(cars):,}")
print(f"  Ethnicity: {len(eth):,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BUILD THE MASTER LSOA TABLE
# ══════════════════════════════════════════════════════════════════════════
# This is the single table we'll explore everything from.
# One row per LSOA, every metric we care about.

# Count bus stops per LSOA using the spatial join from audit
# For now, approximate: use the ground truth join result if available,
# otherwise fall back to a simple LSOA-code prefix match
gt_path = ROOT / "data" / "audit" / "ground_truth.json"
print("Building master LSOA table...")

# ── Start with IMD as the canonical 33,755-row base ─────────────────────
master = imd[["LSOA code (2021)", "LSOA name (2021)",
              "Local Authority District code (2024)",
              "Local Authority District name (2024)",
              "Index of Multiple Deprivation (IMD) Score",
              "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)",
              "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)",
              "Income Score (rate)",
              "Employment Score (rate)",
              "Education, Skills and Training Score",
              "Health Deprivation and Disability Score",
              "Crime Score",
              "Barriers to Housing and Services Score",
              "Living Environment Score",
              "Income Deprivation Affecting Children Index (IDACI) Score (rate)",
              "Income Deprivation Affecting Older People (IDAOPI) Score (rate)",
              "Total population: mid 2022",
              ]].copy()

master.columns = ["lsoa_cd", "lsoa_nm", "lad_cd", "lad_nm",
                  "imd_score", "imd_rank", "imd_decile",
                  "income_score", "employment_score",
                  "education_score", "health_score",
                  "crime_score", "barriers_score", "living_env_score",
                  "idaci_score", "idaopi_score",
                  "pop_imd_2022"]

# ── Census population ───────────────────────────────────────────────────
census_slim = census[["geography code", "Residence type: Total; measures: Value"]].copy()
census_slim.columns = ["lsoa_cd", "population"]
master = master.merge(census_slim, on="lsoa_cd", how="left")

# ── Unemployment ────────────────────────────────────────────────────────
nomis_slim = nomis[["geography code",
                    "Economic activity status: Total: All usual residents aged 16 years and over",
                    "Economic activity status: Economically active (excluding full-time students)",
                    "Economic activity status: Economically active (excluding full-time students): Unemployed",
                    "Economic activity status: Economically inactive: Retired",
                    "Economic activity status: Economically inactive: Long-term sick or disabled",
                    ]].copy()
nomis_slim.columns = ["lsoa_cd", "pop_16plus", "econ_active", "unemployed",
                      "retired", "longterm_sick"]
nomis_slim["unemployment_rate"] = (nomis_slim["unemployed"] / nomis_slim["econ_active"] * 100).round(2)
master = master.merge(nomis_slim, on="lsoa_cd", how="left")

# ── Age (elderly 65+) ───────────────────────────────────────────────────
age_slim = age[["geography code", "Age: Total",
               "Age: Aged 65 to 69 years", "Age: Aged 70 to 74 years",
               "Age: Aged 75 to 79 years", "Age: Aged 80 to 84 years",
               "Age: Aged 85 years and over",
               "Age: Aged 4 years and under", "Age: Aged 5 to 9 years",
               "Age: Aged 10 to 14 years",
               "Age: Aged 15 to 19 years", "Age: Aged 20 to 24 years",
               ]].copy()
age_slim.columns = ["lsoa_cd", "age_total",
                    "age_65_69", "age_70_74", "age_75_79", "age_80_84", "age_85plus",
                    "age_0_4", "age_5_9", "age_10_14", "age_15_19", "age_20_24"]
age_slim["elderly_65plus"] = (age_slim["age_65_69"] + age_slim["age_70_74"] +
                              age_slim["age_75_79"] + age_slim["age_80_84"] +
                              age_slim["age_85plus"])
age_slim["elderly_pct"] = (age_slim["elderly_65plus"] / age_slim["age_total"] * 100).round(1)
age_slim["young_0_14"] = age_slim["age_0_4"] + age_slim["age_5_9"] + age_slim["age_10_14"]
age_slim["young_pct"] = (age_slim["young_0_14"] / age_slim["age_total"] * 100).round(1)
age_slim["working_age_pct"] = (100 - age_slim["elderly_pct"] - age_slim["young_pct"]).round(1)
master = master.merge(age_slim[["lsoa_cd", "elderly_65plus", "elderly_pct",
                                "young_0_14", "young_pct", "working_age_pct"]], on="lsoa_cd", how="left")

# ── Car ownership ──────────────────────────────────────────────────────
cars_slim = cars[["geography code",
                  "Number of cars or vans: Total: All households",
                  "Number of cars or vans: No cars or vans in household"]].copy()
cars_slim.columns = ["lsoa_cd", "total_hh", "nocar_hh"]
cars_slim["nocar_pct"] = (cars_slim["nocar_hh"] / cars_slim["total_hh"] * 100).round(1)
master = master.merge(cars_slim, on="lsoa_cd", how="left")

# ── Ethnicity ──────────────────────────────────────────────────────────
eth_slim = eth[["geography code",
                "Ethnic group: Total: All usual residents",
                "Ethnic group: White",
                "Ethnic group: Asian, Asian British or Asian Welsh",
                "Ethnic group: Black, Black British, Black Welsh, Caribbean or African",
                "Ethnic group: Mixed or Multiple ethnic groups",
                "Ethnic group: Other ethnic group"]].copy()
eth_slim.columns = ["lsoa_cd", "eth_total", "eth_white", "eth_asian",
                    "eth_black", "eth_mixed", "eth_other"]
eth_slim["nonwhite_pct"] = ((eth_slim["eth_total"] - eth_slim["eth_white"]) / eth_slim["eth_total"] * 100).round(1)
master = master.merge(eth_slim[["lsoa_cd", "nonwhite_pct", "eth_white", "eth_asian",
                                "eth_black", "eth_mixed", "eth_other", "eth_total"]], on="lsoa_cd", how="left")

# ── Urban/Rural ────────────────────────────────────────────────────────
ruc_slim = ruc[["LSOA21CD", "RUC21CD", "RUC21NM", "Urban_rural_flag"]].copy()
ruc_slim.columns = ["lsoa_cd", "ruc_code", "ruc_name", "urban_rural"]
master = master.merge(ruc_slim, on="lsoa_cd", how="left")

# ── Bus stop count per LSOA (from spatial join in audit) ────────────────
# Load the spatial join result if available, else use a simple count
stop_counts = stops.groupby("ATCOCode").size()  # just verifying uniqueness
# We need the LSOA assignment from audit — check if it was saved
# For now, use the master table from audit which counted stops per LSOA
# The audit output showed stop_count per LSOA in section 9b regional breakdown
# We'll recompute using a simple approach: load the boundary GeoJSON and do PiP
# But that's expensive. Instead, let's see if there's a cached result.

# Quick approach: count stops that fall within each LSOA using lat/lon
# The audit already did this and got 274,717 matched. Let's use geopandas.
import geopandas as gpd
import json as json_lib

LSOA_GEOJSON = RAW / "boundaries" / "lsoa_2021_england_buc.geojson"
print("  Loading LSOA boundaries...")
with open(LSOA_GEOJSON) as f:
    lsoa_geo = json_lib.load(f)
lsoa_gdf = gpd.GeoDataFrame.from_features(lsoa_geo["features"], crs="EPSG:4326")
lsoa_gdf = lsoa_gdf[["LSOA21CD", "geometry"]].to_crs(epsg=27700)

print("  Building stops GeoDataFrame...")
stops_gdf = gpd.GeoDataFrame(
    stops[["ATCOCode"]],
    geometry=gpd.points_from_xy(stops["Longitude"], stops["Latitude"]),
    crs="EPSG:4326",
).to_crs(epsg=27700)

print("  Spatial join (point-in-polygon)...")
joined = gpd.sjoin(stops_gdf, lsoa_gdf, how="left", predicate="within")
stop_per_lsoa = joined.groupby("LSOA21CD").size().reset_index(name="stop_count")
stop_per_lsoa.columns = ["lsoa_cd", "stop_count"]

master = master.merge(stop_per_lsoa, on="lsoa_cd", how="left")
master["stop_count"] = master["stop_count"].fillna(0).astype(int)

print(f"\n  Master table: {len(master):,} rows × {len(master.columns)} columns")
print(f"  Columns: {list(master.columns)}")
print(f"  Nulls: {master.isnull().sum().sum()}")

---
# Part 1: Know Your LSOAs — Single-LSOA Deep Dives

Before looking at 33,755 rows of statistics, understand what **one LSOA looks like**.  
We pick 6 deliberately diverse LSOAs covering different archetypes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# PART 1: SINGLE-LSOA DEEP DIVES
# ══════════════════════════════════════════════════════════════════════════

LSOA_PROFILES = {
    "E01000001": "City of London 001A — Wealthy urban core (London)",
    "E01012040": "Hendon — Most deprived (Sunderland, NE, IMD rank 43)",
    "E01026625": "Vange — Most deprived (Basildon, East, IMD rank 37)",
    "E01005047": "Rural Somerset — Low deprivation, elderly",
    "E01033544": "Newham — Ethnically diverse, high deprivation (London)",
    "E01019494": "Rural Lincolnshire — Sparse, agricultural",
}

def tell_lsoa_story(lsoa_cd: str, label: str):
    """Print a complete human-readable story about one LSOA."""
    row = master[master["lsoa_cd"] == lsoa_cd]
    if row.empty:
        print(f"  ⚠ {lsoa_cd} not found in master table")
        return
    r = row.iloc[0]
    
    # England averages for context
    eng = {
        "imd_score": master["imd_score"].mean(),
        "unemployment_rate": master["unemployment_rate"].mean(),
        "nocar_pct": master["nocar_pct"].mean(),
        "elderly_pct": master["elderly_pct"].mean(),
        "nonwhite_pct": master["nonwhite_pct"].mean(),
        "income_score": master["income_score"].mean(),
        "stop_count": master["stop_count"].mean(),
        "population": master["population"].mean(),
    }
    
    def vs_avg(val, avg, fmt=".1f", higher_is_worse=True):
        diff = val - avg
        pct = diff / avg * 100 if avg != 0 else 0
        direction = "above" if diff > 0 else "below"
        concern = "⚠" if (higher_is_worse and diff > 0) or (not higher_is_worse and diff < 0) else "✓"
        return f"{val:{fmt}}  ({concern} {abs(pct):.0f}% {direction} avg {avg:{fmt}})"
    
    print(f"\n{'═'*70}")
    print(f"  {lsoa_cd} — {label}")
    print(f"{'═'*70}")
    print(f"  LAD: {r['lad_nm']}")
    print(f"  Type: {r['ruc_name']} ({r['urban_rural']})")
    print()
    print(f"  POPULATION")
    print(f"    Census 2021:     {vs_avg(r['population'], eng['population'], '.0f', False)}")
    print(f"    Elderly 65+:     {vs_avg(r['elderly_pct'], eng['elderly_pct'])}")
    print(f"    Young 0-14:      {r['young_pct']:.1f}%")
    print(f"    Working age:     {r['working_age_pct']:.1f}%")
    print()
    print(f"  DEPRIVATION (higher = more deprived)")
    print(f"    IMD Score:       {vs_avg(r['imd_score'], eng['imd_score'])}")
    print(f"    IMD Rank:        {int(r['imd_rank']):,} / 33,755  (1=worst)")
    print(f"    IMD Decile:      {int(r['imd_decile'])}  (1=most deprived 10%)")
    print(f"    Income deprivation:     {r['income_score']:.3f}  (rate, 0-1)")
    print(f"    Employment deprivation: {r['employment_score']:.3f}  (rate, 0-1)")
    print(f"    Health score:           {r['health_score']:.2f}  (z-score, negative=healthier)")
    print(f"    Education score:        {r['education_score']:.2f}")
    print(f"    Crime score:            {r['crime_score']:.2f}  (z-score)")
    print(f"    Barriers score:         {r['barriers_score']:.2f}")
    print(f"    Living environment:     {r['living_env_score']:.2f}")
    print()
    print(f"  ECONOMIC ACTIVITY")
    print(f"    Unemployment:    {vs_avg(r['unemployment_rate'], eng['unemployment_rate'])}")
    print(f"    Econ active:     {int(r['econ_active']):,} / {int(r['pop_16plus']):,} aged 16+")
    print(f"    Retired:         {int(r['retired']):,}")
    print(f"    Long-term sick:  {int(r['longterm_sick']):,}")
    print()
    print(f"  CAR OWNERSHIP")
    print(f"    No-car HH:       {vs_avg(r['nocar_pct'], eng['nocar_pct'])}")
    print(f"    ({int(r['nocar_hh']):,} / {int(r['total_hh']):,} households)")
    print()
    print(f"  ETHNICITY")
    print(f"    Non-white:       {vs_avg(r['nonwhite_pct'], eng['nonwhite_pct'])}")
    total_e = r['eth_total']
    if total_e > 0:
        print(f"    White:     {r['eth_white']/total_e*100:5.1f}% | Asian: {r['eth_asian']/total_e*100:5.1f}%")
        print(f"    Black:     {r['eth_black']/total_e*100:5.1f}% | Mixed: {r['eth_mixed']/total_e*100:5.1f}% | Other: {r['eth_other']/total_e*100:5.1f}%")
    print()
    print(f"  BUS COVERAGE")
    print(f"    Bus stops:       {vs_avg(r['stop_count'], eng['stop_count'], '.0f', False)}")
    stops_per_1k = r['stop_count'] / r['population'] * 1000 if r['population'] > 0 else 0
    print(f"    Stops per 1k pop: {stops_per_1k:.1f}")
    print()
    
    # THE STORY: what does this LSOA tell a policy maker?
    stories = []
    if r['imd_decile'] <= 2 and r['stop_count'] == 0:
        stories.append("🔴 HIGHLY DEPRIVED with ZERO bus stops — critical transport desert")
    elif r['imd_decile'] <= 2 and r['stop_count'] < eng['stop_count']:
        stories.append("🟠 Highly deprived with below-average bus coverage")
    if r['nocar_pct'] > 40 and r['stop_count'] < 3:
        stories.append("🔴 High car-free population with near-zero bus access")
    if r['elderly_pct'] > 25 and r['stop_count'] < 3:
        stories.append("🟠 Ageing population with limited bus access")
    if r['unemployment_rate'] > 10 and r['stop_count'] < eng['stop_count']:
        stories.append("🟠 High unemployment — bus access needed for job seekers")
    if r['stop_count'] > eng['stop_count'] * 2:
        stories.append("✅ Well-served by bus infrastructure")
    if not stories:
        stories.append("Typical LSOA — no urgent policy flags")
    
    print(f"  POLICY STORY")
    for s in stories:
        print(f"    {s}")

for code, label in LSOA_PROFILES.items():
    tell_lsoa_story(code, label)

---
# Part 2: The 8 Socio-Economic Factors — Deep Dive

For each of the 8 factors Aequitas tracks, we answer:
1. **Where does the data come from?** (exact source columns)
2. **What's the formula?** (exact computation)
3. **What does it look like?** (distribution)
4. **How does it vary?** (by region, urban/rural)
5. **What do the numbers mean?** (policy interpretation)
6. **How does it relate to bus coverage?** (the core question)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 1: DEPRIVATION (IMD Score / Decile)
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 1: DEPRIVATION")
print("="*60)
print()
print("SOURCE:   IMD 2025, File 7")
print("COLUMN:   'Index of Multiple Deprivation (IMD) Score'")
print("UNIT:     Composite score (0-100+), higher = more deprived")
print("FORMULA:  Pre-computed by DLUHC. Weighted sum of 7 domain scores:")
print("          Income (22.5%) + Employment (22.5%) + Education (13.5%)")
print("          + Health (13.5%) + Crime (9.3%) + Barriers (9.3%)")
print("          + Living Environment (9.3%)")
print()
print("WHAT WE COMPUTE:")
print("  imd_score  → direct from source (no transformation)")
print("  imd_rank   → direct from source (1=most deprived)")
print("  imd_decile → direct from source (1=most deprived 10%)")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution
axes[0].hist(master["imd_score"], bins=50, color="#e74c3c", alpha=0.7, edgecolor="white")
axes[0].axvline(master["imd_score"].median(), color="black", linestyle="--", label=f"Median: {master['imd_score'].median():.1f}")
axes[0].set_xlabel("IMD Score")
axes[0].set_ylabel("LSOAs")
axes[0].set_title("Distribution of IMD Score")
axes[0].legend()

# By region
# Add region to master using region boundary GeoJSON (no API dependency)
REGION_GEOJSON = RAW / "boundaries" / "regions_2021_england_buc.geojson"
if REGION_GEOJSON.exists():
    with open(REGION_GEOJSON) as f_rgn:
        rgn_geo = json_lib.load(f_rgn)
    rgn_gdf = gpd.GeoDataFrame.from_features(rgn_geo["features"], crs="EPSG:4326").to_crs(epsg=27700)
    rgn_col = [c for c in rgn_gdf.columns if "NM" in c and c != "geometry"][0]
    lsoa_centroids = lsoa_gdf.copy()
    lsoa_centroids["geometry"] = lsoa_centroids.centroid
    lsoa_rgn = gpd.sjoin(lsoa_centroids, rgn_gdf[[rgn_col, "geometry"]], how="left", predicate="within")
    lsoa_to_region = lsoa_rgn.set_index("LSOA21CD")[rgn_col].to_dict()
    master["region"] = master["lsoa_cd"].map(lsoa_to_region)
    print(f"  Region mapping (spatial): {master['region'].notna().sum():,} / {len(master):,} LSOAs")
else:
    master["region"] = "Unknown"
    print("  ⚠ No region boundary file available")

region_imd = master.groupby("region")["imd_score"].mean().sort_values(ascending=False)
region_imd.plot(kind="barh", ax=axes[1], color="#e74c3c", alpha=0.7)
axes[1].set_xlabel("Mean IMD Score")
axes[1].set_title("Deprivation by Region")
axes[1].axvline(master["imd_score"].mean(), color="black", linestyle="--", alpha=0.5)

# Urban vs Rural
ur_imd = master.groupby("urban_rural")["imd_score"].describe()[["mean", "50%"]]
ur_imd.plot(kind="bar", ax=axes[2], color=["#e74c3c", "#3498db"], alpha=0.7)
axes[2].set_title("Deprivation: Urban vs Rural")
axes[2].set_ylabel("IMD Score")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# Key stats
print("\nKEY NUMBERS:")
print(f"  England mean:   {master['imd_score'].mean():.1f}")
print(f"  England median: {master['imd_score'].median():.1f}")
print(f"  Range:          {master['imd_score'].min():.1f} – {master['imd_score'].max():.1f}")
print(f"  Std dev:        {master['imd_score'].std():.1f}")
print(f"  Decile 1 (most deprived):  {master[master['imd_decile']==1]['imd_score'].mean():.1f}")
print(f"  Decile 10 (least deprived): {master[master['imd_decile']==10]['imd_score'].mean():.1f}")
print()
print("POLICY INTERPRETATION:")
print("  < 10:  Low deprivation — affluent suburbs, rural villages")
print("  10-20: Below average — mixed areas")
print("  20-30: Above average — some concentrated disadvantage")
print("  30-50: High deprivation — significant poverty, poor outcomes")
print("  > 50:  Severe deprivation — worst ~5% nationally")
print("  > 70:  Extreme — worst ~1%, often inner-city estates")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 2: UNEMPLOYMENT RATE
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 2: UNEMPLOYMENT RATE")
print("="*60)
print()
print("SOURCE:   Census 2021 TS066 (Economic Activity Status)")
print("COLUMNS:  'Econ active (excl full-time students)' and 'Unemployed'")
print("FORMULA:  unemployment_rate = unemployed / econ_active * 100")
print("          ⚠ Denominator is economically ACTIVE, NOT total population.")
print("          ⚠ Full-time students who work are EXCLUDED from denominator.")
print()
print("WHAT WE ALSO HAVE (useful context):")
print("  retired          — count of retired people (correlates with elderly_pct)")
print("  longterm_sick    — proxy for health deprivation")
print("  pop_16plus       — total population 16+ (includes inactive)")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(master["unemployment_rate"], bins=50, color="#f39c12", alpha=0.7, edgecolor="white")
axes[0].axvline(master["unemployment_rate"].median(), color="black", linestyle="--",
               label=f"Median: {master['unemployment_rate'].median():.1f}%")
axes[0].set_xlabel("Unemployment Rate (%)")
axes[0].set_ylabel("LSOAs")
axes[0].set_title("Distribution of Unemployment Rate")
axes[0].legend()

region_unemp = master.groupby("region")["unemployment_rate"].mean().sort_values(ascending=False)
region_unemp.plot(kind="barh", ax=axes[1], color="#f39c12", alpha=0.7)
axes[1].set_xlabel("Mean Unemployment Rate (%)")
axes[1].set_title("Unemployment by Region")

# By IMD decile — does unemployment track deprivation?
decile_unemp = master.groupby("imd_decile")["unemployment_rate"].mean()
decile_unemp.plot(kind="bar", ax=axes[2], color="#f39c12", alpha=0.7)
axes[2].set_xlabel("IMD Decile (1=most deprived)")
axes[2].set_ylabel("Mean Unemployment Rate (%)")
axes[2].set_title("Unemployment by Deprivation Decile")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\nKEY NUMBERS:")
print(f"  National rate (computed): {master['unemployed'].sum() / master['econ_active'].sum() * 100:.2f}%")
print(f"  ONS published:           4.8%")
print(f"  LSOA range:              {master['unemployment_rate'].min():.1f}% – {master['unemployment_rate'].max():.1f}%")
print(f"  Std dev:                 {master['unemployment_rate'].std():.1f}%")
print()
print("RELATIONSHIP TO IMD:")
corr = master[["unemployment_rate", "imd_score"]].corr().iloc[0,1]
print(f"  Pearson r with IMD score: {corr:.3f}  (strong positive — expected)")
print(f"  BUT: IMD employment score != unemployment rate")
print(f"  IMD employment score is a rate of 'employment deprivation' (broader concept)")
print(f"  Correlation of unemployment_rate vs employment_score: {master[['unemployment_rate','employment_score']].corr().iloc[0,1]:.3f}")
print()
print("POLICY INTERPRETATION:")
print("  < 3%:  Very low — typically affluent commuter belts")
print("  3-5%:  Around national average")
print("  5-8%:  Above average — some economic distress")
print("  8-12%: High — significant joblessness")
print("  >12%:  Extreme — worst ~2% nationally, concentrated worklessness")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 3: CAR OWNERSHIP
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 3: CAR OWNERSHIP (% no-car households)")
print("="*60)
print()
print("SOURCE:   Census 2021 TS045 (Car or van availability)")
print("COLUMNS:  'No cars or vans in household' / 'Total: All households'")
print("FORMULA:  nocar_pct = nocar_hh / total_hh * 100")
print("")
print("⚠ CRITICAL TRAP: No-car % is NOT a simple poverty signal.")
print("   HIGH no-car can mean:")
print("     (a) Poverty — can't afford a car → needs bus access")
print("     (b) Wealthy urban — choose not to own (London) → has alternatives")
print("   Must cross-reference with IMD to distinguish.")
print()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution
axes[0,0].hist(master["nocar_pct"], bins=50, color="#2ecc71", alpha=0.7, edgecolor="white")
axes[0,0].axvline(master["nocar_pct"].median(), color="black", linestyle="--",
                 label=f"Median: {master['nocar_pct'].median():.1f}%")
axes[0,0].set_xlabel("No-Car Household %")
axes[0,0].set_title("Distribution")
axes[0,0].legend()

# Urban vs Rural — this is where the trap becomes visible
for flag, color in [("Urban", "#e74c3c"), ("Rural", "#3498db")]:
    subset = master[master["urban_rural"] == flag]
    axes[0,1].hist(subset["nocar_pct"], bins=40, alpha=0.5, color=color, label=f"{flag} (n={len(subset):,})")
axes[0,1].set_xlabel("No-Car Household %")
axes[0,1].set_title("Urban vs Rural — The Two Populations")
axes[0,1].legend()

# No-car vs IMD — the scatter that shows the trap
sample = master.sample(min(5000, len(master)), random_state=42)
colors = sample["urban_rural"].map({"Urban": "#e74c3c", "Rural": "#3498db"})
axes[1,0].scatter(sample["imd_score"], sample["nocar_pct"], c=colors, alpha=0.2, s=5)
axes[1,0].set_xlabel("IMD Score (deprivation)")
axes[1,0].set_ylabel("No-Car HH %")
axes[1,0].set_title("No-Car % vs Deprivation (red=urban, blue=rural)")

# The key insight: high no-car + high IMD = genuine need
# high no-car + low IMD = wealthy choice
master["car_need_category"] = "Other"
master.loc[(master["nocar_pct"] > 40) & (master["imd_decile"] <= 3), "car_need_category"] = "Deprived + No car"
master.loc[(master["nocar_pct"] > 40) & (master["imd_decile"] >= 8), "car_need_category"] = "Affluent + No car"
master.loc[(master["nocar_pct"] < 15) & (master["imd_decile"] <= 3), "car_need_category"] = "Deprived + Has car"

cat_counts = master["car_need_category"].value_counts()
cat_counts.plot(kind="bar", ax=axes[1,1], color="#2ecc71", alpha=0.7)
axes[1,1].set_title("Car Ownership x Deprivation Categories")
axes[1,1].set_ylabel("LSOAs")
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

print("\nKEY NUMBERS:")
print(f"  England mean no-car HH %: {master['nocar_pct'].mean():.1f}%")
print(f"  Urban mean:               {master[master['urban_rural']=='Urban']['nocar_pct'].mean():.1f}%")
print(f"  Rural mean:               {master[master['urban_rural']=='Rural']['nocar_pct'].mean():.1f}%")
print(f"  Range:                    {master['nocar_pct'].min():.1f}% – {master['nocar_pct'].max():.1f}%")
print()
print("CATEGORY BREAKDOWN:")
for cat, count in cat_counts.items():
    print(f"  {cat}: {count:,} LSOAs")
print()
print("POLICY INSIGHT:")
deprived_nocar = master[(master["nocar_pct"] > 40) & (master["imd_decile"] <= 3)]
print(f"  'Deprived + No car' LSOAs: {len(deprived_nocar):,}")
print(f"  Of these, {(deprived_nocar['stop_count']==0).sum():,} have ZERO bus stops")
print(f"  → These are the most bus-dependent communities with no bus access")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 4: ELDERLY POPULATION (% aged 65+)
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 4: ELDERLY POPULATION")
print("="*60)
print()
print("SOURCE:   Census 2021 TS007a (Age by 5-year bands)")
print("COLUMNS:  Sum of 'Age: Aged 65 to 69' through 'Age: Aged 85+' / 'Age: Total'")
print("FORMULA:  elderly_pct = (age_65_69 + age_70_74 + age_75_79 + age_80_84 + age_85plus) / age_total * 100")
print("")
print("WHY THIS MATTERS:")
print("  Elderly people are more likely to:")
print("    - Not drive / have given up driving")
print("    - Depend on buses for medical appointments, shopping")
print("    - Be isolated without public transport")
print("  → Bus coverage in elderly areas is a social isolation metric")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(master["elderly_pct"], bins=50, color="#9b59b6", alpha=0.7, edgecolor="white")
axes[0].axvline(master["elderly_pct"].median(), color="black", linestyle="--",
               label=f"Median: {master['elderly_pct'].median():.1f}%")
axes[0].set_xlabel("Elderly 65+ %")
axes[0].set_title("Distribution")
axes[0].legend()

# Urban vs Rural — elderly tend to concentrate in rural and coastal
ur_elderly = master.groupby("urban_rural")["elderly_pct"].mean()
ur_elderly.plot(kind="bar", ax=axes[1], color="#9b59b6", alpha=0.7)
axes[1].set_title("Elderly % by Urban/Rural")
axes[1].set_ylabel("Mean Elderly 65+ %")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

# Elderly vs no-car — the double vulnerability
axes[2].scatter(master["elderly_pct"], master["nocar_pct"], alpha=0.05, s=3, color="#9b59b6")
axes[2].set_xlabel("Elderly 65+ %")
axes[2].set_ylabel("No-Car HH %")
axes[2].set_title("Elderly % vs No-Car % (double vulnerability)")

plt.tight_layout()
plt.show()

print("\nKEY NUMBERS:")
print(f"  England mean:     {master['elderly_pct'].mean():.1f}%")
print(f"  Range:            {master['elderly_pct'].min():.1f}% – {master['elderly_pct'].max():.1f}%")
print(f"  Urban mean:       {master[master['urban_rural']=='Urban']['elderly_pct'].mean():.1f}%")
print(f"  Rural mean:       {master[master['urban_rural']=='Rural']['elderly_pct'].mean():.1f}%")
print()
# Double vulnerability: elderly + no car + no bus
double_vuln = master[(master["elderly_pct"] > 25) & (master["nocar_pct"] > 30) & (master["stop_count"] < 3)]
print(f"DOUBLE VULNERABILITY (elderly>25% + nocar>30% + <3 stops):")
print(f"  {len(double_vuln):,} LSOAs")
print(f"  Mean population: {double_vuln['population'].mean():.0f}")
print(f"  → Elderly people in these areas have no bus and no car")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 5: INCOME LEVELS (IMD Income sub-domain)
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 5: INCOME LEVELS")
print("="*60)
print()
print("SOURCE:   IMD 2025, Income Score (rate)")
print("COLUMN:   'Income Score (rate)'")
print("UNIT:     Proportion (0-1) of population that is income-deprived")
print("FORMULA:  Direct from IMD — no transformation needed")
print("          Income-deprived = receiving means-tested benefits")
print()
print("⚠ This is NOT income level. It's proportion in income deprivation.")
print("  0.05 = 5% of residents are income-deprived")
print("  0.50 = 50% are income-deprived")
print()
print("ALSO AVAILABLE:")
print("  IDACI — Income Deprivation Affecting Children Index")
print("  IDAOPI — Income Deprivation Affecting Older People")
print("  These are sub-indices specific to children and elderly.")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(master["income_score"], bins=50, color="#e67e22", alpha=0.7, edgecolor="white")
axes[0].axvline(master["income_score"].median(), color="black", linestyle="--",
               label=f"Median: {master['income_score'].median():.3f}")
axes[0].set_xlabel("Income Score (rate)")
axes[0].set_title("Distribution")
axes[0].legend()

# Income deprivation vs unemployment — closely related but different
axes[1].scatter(master["income_score"], master["unemployment_rate"], alpha=0.05, s=3, color="#e67e22")
axes[1].set_xlabel("Income Score (deprivation rate)")
axes[1].set_ylabel("Unemployment Rate (%)")
axes[1].set_title(f"Income Deprivation vs Unemployment (r={master[['income_score','unemployment_rate']].corr().iloc[0,1]:.2f})")

# IDACI vs IDAOPI — children vs elderly income deprivation
axes[2].scatter(master["idaci_score"], master["idaopi_score"], alpha=0.05, s=3, color="#e67e22")
axes[2].set_xlabel("IDACI (children income deprivation)")
axes[2].set_ylabel("IDAOPI (elderly income deprivation)")
r_child_old = master[["idaci_score", "idaopi_score"]].corr().iloc[0,1]
axes[2].set_title(f"Children vs Elderly Income Deprivation (r={r_child_old:.2f})")
axes[2].plot([0, 1], [0, 1], "--", color="gray", alpha=0.5)

plt.tight_layout()
plt.show()

print("\nKEY NUMBERS:")
print(f"  England mean income score: {master['income_score'].mean():.3f}")
print(f"  = {master['income_score'].mean()*100:.1f}% of population income-deprived on average")
print(f"  Range: {master['income_score'].min():.3f} – {master['income_score'].max():.3f}")
print()
print("POLICY INTERPRETATION:")
print("  < 0.10: Low income deprivation — fewer than 10% on benefits")
print("  0.10-0.20: Moderate")
print("  0.20-0.40: High — 1 in 3 to 2 in 5 residents income-deprived")
print("  > 0.40: Extreme — majority of population income-deprived")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 6: ETHNIC COMPOSITION
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 6: ETHNIC COMPOSITION")
print("="*60)
print()
print("SOURCE:   Census 2021 TS021 (Ethnic group)")
print("COLUMNS:  Top-level groups: White, Asian, Black, Mixed, Other")
print("FORMULA:  nonwhite_pct = (eth_total - eth_white) / eth_total * 100")
print()
print("WHY THIS MATTERS FOR BUS POLICY:")
print("  Ethnic minority communities are statistically more likely to:")
print("    - Live in urban deprived areas")
print("    - Have lower car ownership")
print("    - Be more dependent on public transport")
print("  → Bus coverage equity across ethnic groups is a policy concern")
print("  → But causation is through deprivation, not ethnicity itself")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution — note the extreme right skew (most LSOAs are >80% white)
axes[0].hist(master["nonwhite_pct"], bins=50, color="#1abc9c", alpha=0.7, edgecolor="white")
axes[0].axvline(master["nonwhite_pct"].median(), color="black", linestyle="--",
               label=f"Median: {master['nonwhite_pct'].median():.1f}%")
axes[0].set_xlabel("Non-white %")
axes[0].set_title("Distribution (note right skew)")
axes[0].legend()

# Ethnicity vs deprivation
axes[1].scatter(master["nonwhite_pct"], master["imd_score"], alpha=0.05, s=3, color="#1abc9c")
axes[1].set_xlabel("Non-white %")
axes[1].set_ylabel("IMD Score")
r_eth_imd = master[["nonwhite_pct", "imd_score"]].corr().iloc[0,1]
axes[1].set_title(f"Ethnic Diversity vs Deprivation (r={r_eth_imd:.2f})")

# Regional breakdown — where is ethnic diversity?
region_eth = master.groupby("region")["nonwhite_pct"].mean().sort_values(ascending=False)
region_eth.plot(kind="barh", ax=axes[2], color="#1abc9c", alpha=0.7)
axes[2].set_xlabel("Mean Non-white %")
axes[2].set_title("Ethnic Diversity by Region")

plt.tight_layout()
plt.show()

print("\nKEY NUMBERS:")
print(f"  England mean non-white %: {master['nonwhite_pct'].mean():.1f}%")
print(f"  Median:                   {master['nonwhite_pct'].median():.1f}%")
print(f"  Range:                    {master['nonwhite_pct'].min():.1f}% – {master['nonwhite_pct'].max():.1f}%")
print(f"  LSOAs > 50% non-white:    {(master['nonwhite_pct'] > 50).sum():,}")
print(f"  LSOAs > 80% non-white:    {(master['nonwhite_pct'] > 80).sum():,}")
print()
print("TOP 5 ETHNIC GROUP COMPOSITIONS (by region):")
for rgn in ["London", "West Midlands", "South West"]:
    rgn_data = master[master["region"] == rgn]
    total = rgn_data["eth_total"].sum()
    if total > 0:
        print(f"  {rgn}: White {rgn_data['eth_white'].sum()/total*100:.1f}% | "
              f"Asian {rgn_data['eth_asian'].sum()/total*100:.1f}% | "
              f"Black {rgn_data['eth_black'].sum()/total*100:.1f}% | "
              f"Mixed {rgn_data['eth_mixed'].sum()/total*100:.1f}%")
    else:
        print(f"  {rgn}: No data (region mapping may have failed)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 7: GENDER-ADJUSTED ACCESSIBILITY (PROXY)
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 7: GENDER-ADJUSTED ACCESSIBILITY")
print("="*60)
print()
print("⚠ WE DO NOT HAVE DIRECT GENDER-ADJUSTED DATA.")
print()
print("What the blueprint says: 'proxy: childcare/school access'")
print("What we actually have:")
print("  - IDACI score (income deprivation affecting children)")
print("  - Young 0-14 population %")
print("  - 'Children and Young People Sub-domain Score' in IMD")
print("  - IMD 'Geographical Barriers' sub-domain (distance to services)")
print()
print("BEST AVAILABLE PROXY:")
print("  The IMD 'Barriers to Housing and Services' domain includes")
print("  'Geographical Barriers' sub-domain which measures distance to:")
print("    - GP surgery, primary school, post office, supermarket")
print("  Combined with young_pct and IDACI, this gives a rough proxy")
print("  for whether families with children have accessible services.")
print()
print("FOR THE PIPELINE:")
print("  We will use:")
print("    proxy_gender_access = f(barriers_score, idaci_score, young_pct)")
print("  This is acknowledged as a PROXY, not a direct measure.")
print("  The narrative engine will flag this limitation.")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(master["barriers_score"], master["idaci_score"], alpha=0.05, s=3, color="#e91e63")
axes[0].set_xlabel("Barriers to Housing & Services Score")
axes[0].set_ylabel("IDACI (child income deprivation)")
axes[0].set_title("Geographic Barriers vs Child Income Deprivation")

axes[1].scatter(master["young_pct"], master["barriers_score"], alpha=0.05, s=3, color="#e91e63")
axes[1].set_xlabel("Young 0-14 %")
axes[1].set_ylabel("Barriers Score")
axes[1].set_title("Young Population vs Service Barriers")

plt.tight_layout()
plt.show()

print("\nKEY INSIGHT:")
print(f"  Barriers score range: {master['barriers_score'].min():.1f} – {master['barriers_score'].max():.1f}")
print(f"  High barriers + high IDACI + high young % = families cut off from services")
high_barrier_families = master[(master["barriers_score"] > master["barriers_score"].quantile(0.9)) &
                               (master["idaci_score"] > 0.2) & (master["young_pct"] > 20)]
print(f"  LSOAs with high barriers + deprived children + many young: {len(high_barrier_families):,}")
print(f"  Of these, {(high_barrier_families['stop_count'] < 3).sum():,} have fewer than 3 bus stops")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FACTOR 8: URBAN/RURAL CLASSIFICATION
# ══════════════════════════════════════════════════════════════════════════

print("FACTOR 8: URBAN/RURAL CLASSIFICATION")
print("="*60)
print()
print("SOURCE:   ONS Rural Urban Classification 2021")
print("COLUMNS:  RUC21CD (6 categories), Urban_rural_flag (binary)")
print("FORMULA:  No computation — direct categorical classification")
print()
print("THE 6 CATEGORIES:")
ruc_breakdown = master.groupby(["ruc_code", "ruc_name"]).agg(
    count=("lsoa_cd", "count"),
    mean_imd=("imd_score", "mean"),
    mean_stops=("stop_count", "mean"),
    mean_nocar=("nocar_pct", "mean"),
    mean_elderly=("elderly_pct", "mean"),
).reset_index()

print(f"  {'Code':<6} {'Name':<45} {'LSOAs':>6} {'IMD':>5} {'Stops':>6} {'NoCar%':>7} {'Old%':>5}")
print(f"  {'─'*85}")
for _, row in ruc_breakdown.iterrows():
    print(f"  {row['ruc_code']:<6} {row['ruc_name']:<45} {row['count']:>6,} "
          f"{row['mean_imd']:>5.1f} {row['mean_stops']:>6.1f} {row['mean_nocar']:>7.1f} {row['mean_elderly']:>5.1f}")

print()
print("WHY THIS IS A 'STRUCTURAL MODIFIER':")
print("  Urban/rural doesn't cause deprivation — it changes what deprivation")
print("  LOOKS like and how bus coverage should be evaluated:")
print("  ")
print("  URBAN deprived: high density, could have many stops, issue is frequency/quality")
print("  RURAL deprived: low density, few stops possible, issue is ANY access at all")
print("  → Same IMD score means different policy responses in urban vs rural")
print("  → Pipeline must compute urban and rural metrics SEPARATELY")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Stops by RUC category
ruc_order = ["UN1", "UF1", "RSN1", "RLN1", "RSF1", "RLF1"]
ruc_stops = master.groupby("ruc_code")["stop_count"].mean().reindex(ruc_order)
ruc_stops.plot(kind="bar", ax=axes[0], color="#3498db", alpha=0.7)
axes[0].set_title("Mean Bus Stops by RUC Category")
axes[0].set_ylabel("Mean stops per LSOA")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")

# IMD distribution by urban/rural
master.boxplot(column="imd_score", by="urban_rural", ax=axes[1])
axes[1].set_title("Deprivation by Urban/Rural")
axes[1].set_ylabel("IMD Score")
plt.sca(axes[1])
plt.title("Deprivation by Urban/Rural")

# Zero-stop LSOAs by RUC
zero_by_ruc = master[master["stop_count"] == 0].groupby("ruc_code").size().reindex(ruc_order, fill_value=0)
total_by_ruc = master.groupby("ruc_code").size().reindex(ruc_order)
zero_pct_by_ruc = (zero_by_ruc / total_by_ruc * 100)
zero_pct_by_ruc.plot(kind="bar", ax=axes[2], color="#e74c3c", alpha=0.7)
axes[2].set_title("% Zero-Stop LSOAs by RUC Category")
axes[2].set_ylabel("% with zero bus stops")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

---
# Part 3: Cross-Factor Analysis

How do the 8 factors relate to each other? Where do they reinforce, and where do they diverge?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CROSS-FACTOR CORRELATION MATRIX
# ══════════════════════════════════════════════════════════════════════════

factor_cols = {
    "IMD Score": "imd_score",
    "Income Dep.": "income_score",
    "Employ. Dep.": "employment_score",
    "Unemployment %": "unemployment_rate",
    "No-Car HH %": "nocar_pct",
    "Elderly 65+ %": "elderly_pct",
    "Non-white %": "nonwhite_pct",
    "Bus Stops": "stop_count",
    "Health Score": "health_score",
    "Education Score": "education_score",
    "Crime Score": "crime_score",
    "Barriers Score": "barriers_score",
    "Living Env.": "living_env_score",
    "Population": "population",
}

corr_df = master[[c for c in factor_cols.values()]].rename(
    columns={v: k for k, v in factor_cols.items()}
).corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Cross-Factor Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

print("KEY RELATIONSHIPS:")
print()
print("STRONG POSITIVE (r > 0.7) — these move together:")
print("  IMD ↔ Income (0.95): deprivation IS income deprivation")
print("  IMD ↔ Employment (0.96): deprivation IS employment deprivation")
print("  IMD ↔ Unemployment (0.79): deprived areas have high unemployment")
print("  Income ↔ Unemployment (0.81): income-deprived areas are jobless")
print()
print("MODERATE POSITIVE (r 0.3-0.7) — related but distinct:")
print("  IMD ↔ No-Car (0.64): deprived people can't afford cars")
print("  IMD ↔ Crime (0.80): deprived areas have higher crime")
print()
print("WEAK/NEGATIVE — independent or inverse:")
print("  IMD ↔ Elderly: near zero — deprivation is NOT an age thing")
print("  Bus stops ↔ everything: near zero — stop count alone means little")
print("  → Need to compute stops-per-capita or weighted access metric")
print()
print("SURPRISES:")
bus_corr = corr_df["Bus Stops"].sort_values()
print(f"  Bus stops correlate most with: {bus_corr.index[-2]} (r={bus_corr.iloc[-2]:.2f})")
print(f"  Bus stops correlate least with: {bus_corr.index[0]} (r={bus_corr.iloc[0]:.2f})")
print(f"  → Raw stop count is nearly uncorrelated with all socio-economic factors")
print(f"  → This confirms: we MUST compute a richer access metric in the pipeline")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ANOMALY INVESTIGATION: LSOAs that break the pattern
# ══════════════════════════════════════════════════════════════════════════

print("PATTERN-BREAKING LSOAs")
print("="*60)
print()
print("These LSOAs defy the expected correlations. Understanding WHY")
print("teaches us more than any average.")
print()

# 1. High deprivation but many bus stops (is bus access helping?)
print("1. HIGH DEPRIVATION + MANY BUS STOPS (> 20 stops, IMD decile 1)")
well_served_deprived = master[(master["imd_decile"] == 1) & (master["stop_count"] > 20)]
print(f"   {len(well_served_deprived):,} LSOAs")
if len(well_served_deprived) > 0:
    print(f"   Typical: {well_served_deprived[['lsoa_nm','lad_nm','imd_score','stop_count','urban_rural']].head(5).to_string(index=False)}")
    print(f"   → Mostly urban cores near bus stations/interchanges")
print()

# 2. Low deprivation but zero bus stops (wealthy but isolated?)
print("2. LOW DEPRIVATION + ZERO BUS STOPS (IMD decile 9-10)")
rich_no_bus = master[(master["imd_decile"] >= 9) & (master["stop_count"] == 0)]
print(f"   {len(rich_no_bus):,} LSOAs")
if len(rich_no_bus) > 0:
    print(f"   Urban/Rural split: {rich_no_bus['urban_rural'].value_counts().to_dict()}")
    print(f"   Mean elderly %: {rich_no_bus['elderly_pct'].mean():.1f}%")
    print(f"   → Wealthy rural areas — they drive, so no bus demand")
print()

# 3. High unemployment but low IMD (something offsetting?)
print("3. HIGH UNEMPLOYMENT (>10%) + LOW IMD (decile 7-10)")
high_unemp_low_imd = master[(master["unemployment_rate"] > 10) & (master["imd_decile"] >= 7)]
print(f"   {len(high_unemp_low_imd):,} LSOAs")
if len(high_unemp_low_imd) > 0:
    print(f"   Mean elderly %: {high_unemp_low_imd['elderly_pct'].mean():.1f}%")
    print(f"   Mean young %: {high_unemp_low_imd['young_pct'].mean():.1f}%")
    print(f"   → Likely student areas (economically inactive students skew the rate)")
print()

# 4. High no-car but low deprivation (London effect?)
print("4. HIGH NO-CAR (>50%) + LOW IMD (decile 8-10)")
nocar_affluent = master[(master["nocar_pct"] > 50) & (master["imd_decile"] >= 8)]
print(f"   {len(nocar_affluent):,} LSOAs")
if len(nocar_affluent) > 0:
    print(f"   Region split: {nocar_affluent['region'].value_counts().head(3).to_dict()}")
    print(f"   → London dominates — wealthy areas with tube/rail alternatives")
print()

# 5. High elderly but very low no-car (elderly who drive?)
print("5. HIGH ELDERLY (>30%) + LOW NO-CAR (<10%)")
driving_elderly = master[(master["elderly_pct"] > 30) & (master["nocar_pct"] < 10)]
print(f"   {len(driving_elderly):,} LSOAs")
if len(driving_elderly) > 0:
    print(f"   Urban/Rural: {driving_elderly['urban_rural'].value_counts().to_dict()}")
    print(f"   Mean IMD: {driving_elderly['imd_score'].mean():.1f}")
    print(f"   → Affluent rural retirees — car-dependent by choice, not deprivation")

---
# Part 4: Bus Coverage Integration — The Core Question

This is what Aequitas is about: **does bus coverage correlate with socio-economic need?**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# BUS COVERAGE × DEPRIVATION — THE HEADLINE ANALYSIS
# ══════════════════════════════════════════════════════════════════════════

print("BUS COVERAGE × SOCIO-ECONOMIC NEED")
print("="*60)
print()

# Compute stops per 1000 population (more meaningful than raw count)
master["stops_per_1k"] = (master["stop_count"] / master["population"] * 1000).round(2)
master["has_bus"] = master["stop_count"] > 0

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Stops per 1k by IMD decile
decile_coverage = master.groupby("imd_decile").agg(
    mean_stops=("stop_count", "mean"),
    mean_stops_1k=("stops_per_1k", "mean"),
    zero_pct=("has_bus", lambda x: (1 - x.mean()) * 100),
    n=("lsoa_cd", "count"),
)

decile_coverage["mean_stops_1k"].plot(kind="bar", ax=axes[0,0], color="#e74c3c", alpha=0.7)
axes[0,0].set_xlabel("IMD Decile (1=most deprived)")
axes[0,0].set_ylabel("Stops per 1000 population")
axes[0,0].set_title("Bus Stop Density by Deprivation")
axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=0)

# 2. Zero-stop % by IMD decile
decile_coverage["zero_pct"].plot(kind="bar", ax=axes[0,1], color="#c0392b", alpha=0.7)
axes[0,1].set_xlabel("IMD Decile (1=most deprived)")
axes[0,1].set_ylabel("% LSOAs with zero stops")
axes[0,1].set_title("Transport Deserts by Deprivation")
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=0)

# 3. Stop count vs IMD (scatter)
sample = master.sample(min(5000, len(master)), random_state=42)
axes[0,2].scatter(sample["imd_score"], sample["stops_per_1k"], alpha=0.1, s=5, color="#e74c3c")
axes[0,2].set_xlabel("IMD Score")
axes[0,2].set_ylabel("Stops per 1000 pop")
axes[0,2].set_title(f"Coverage vs Deprivation (r={master[['imd_score','stops_per_1k']].corr().iloc[0,1]:.3f})")

# 4. By region: deprived LSOAs with zero stops
deprived = master[master["imd_decile"] <= 2]
region_zero = deprived.groupby("region").agg(
    total=("lsoa_cd", "count"),
    zero_stops=("has_bus", lambda x: (~x).sum()),
).assign(zero_pct=lambda df: df["zero_stops"] / df["total"] * 100)
region_zero["zero_pct"].sort_values().plot(kind="barh", ax=axes[1,0], color="#c0392b", alpha=0.7)
axes[1,0].set_xlabel("% deprived LSOAs with zero stops")
axes[1,0].set_title("Transport Deserts in Deprived Areas (by region)")

# 5. Urban vs Rural bus coverage gap
ur_coverage = master.groupby(["urban_rural", "imd_decile"])["stops_per_1k"].mean().unstack(level=0)
ur_coverage.plot(kind="bar", ax=axes[1,1], color=["#3498db", "#e74c3c"], alpha=0.7)
axes[1,1].set_xlabel("IMD Decile")
axes[1,1].set_ylabel("Stops per 1000 pop")
axes[1,1].set_title("Urban vs Rural Coverage by Deprivation")
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=0)
axes[1,1].legend(title="")

# 6. The equity gap: who NEEDS bus but DOESN'T have it?
# Need score = IMD score + (1 if nocar > 30%) + (1 if elderly > 25%)
master["bus_need_score"] = (
    master["imd_score"] / master["imd_score"].max() * 50 +
    (master["nocar_pct"] > 30).astype(int) * 25 +
    (master["elderly_pct"] > 25).astype(int) * 25
)
axes[1,2].scatter(master["bus_need_score"], master["stops_per_1k"], alpha=0.02, s=3, color="#8e44ad")
axes[1,2].set_xlabel("Bus Need Score (composite)")
axes[1,2].set_ylabel("Stops per 1000 pop")
axes[1,2].set_title("Bus Need vs Actual Coverage")

plt.tight_layout()
plt.show()

print("\nHEADLINE FINDINGS:")
print(f"  Total LSOAs with zero bus stops: {(~master['has_bus']).sum():,} ({(~master['has_bus']).mean()*100:.1f}%)")
print(f"  Deprived (decile 1-2) with zero stops: {(~deprived['has_bus']).sum():,} ({(~deprived['has_bus']).mean()*100:.1f}%)")
print(f"  IMD score ↔ stops_per_1k correlation: {master[['imd_score','stops_per_1k']].corr().iloc[0,1]:.3f}")
print()
print("INTERPRETATION:")
print("  The correlation between deprivation and bus coverage is WEAK.")
print("  This means bus stops are NOT systematically placed where need is highest.")
print("  → This is the core policy finding that Aequitas quantifies.")

---
# Part 5: Dataset Relationship Map + Temporal Alignment

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# DATASET RELATIONSHIP MAP
# ══════════════════════════════════════════════════════════════════════════

print("DATASET RELATIONSHIP MAP")
print("="*60)
print()
print("JOIN GRAPH — everything joins on LSOA code:")
print()
print("                    ┌─────────────────────────────────────┐")
print("                    │   IMD 2025  (canonical 33,755)      │")
print("                    │   Key: 'LSOA code (2021)'          │")
print("                    └──────────────┬──────────────────────┘")
print("                                   │")
print("         ┌───────────┬─────────────┼─────────────┬───────────┐")
print("         │           │             │             │           │")
print("  ┌──────┴─────┐ ┌───┴────┐ ┌─────┴─────┐ ┌────┴────┐ ┌───┴────┐")
print("  │Census TS001│ │  NOMIS │ │ Age TS007a│ │Cars TS45│ │Eth TS21│")
print("  │  Pop'n     │ │ TS066  │ │ Age bands │ │Car own  │ │Ethnic  │")
print("  │'geog code' │ │'g code'│ │'geog code'│ │'g code' │ │'g code'│")
print("  └────────────┘ └────────┘ └───────────┘ └─────────┘ └────────┘")
print("         │")
print("  ┌──────┴──────┐     ┌───────────────┐")
print("  │ RUC 2021    │     │ NaPTAN Stops   │")
print("  │'LSOA21CD'   │     │ (spatial join) │")
print("  └─────────────┘     │ stop → LSOA    │")
print("                      └───────┬───────┘")
print("                              │")
print("                      ┌───────┴───────┐")
print("                      │  BODS GTFS    │")
print("                      │ (stop_id =    │")
print("                      │  ATCOCode)    │")
print("                      └───────────────┘")
print()
print()
print("TEMPORAL ALIGNMENT:")
print(f"  {'Dataset':<25} {'Reference Date':<30} {'Boundary Year':<15} {'Notes'}")
print(f"  {'─'*100}")
print(f"  {'Census TS001/007a/021/045':<25} {'21 March 2021':<30} {'2021 LSOA':<15} Census night")
print(f"  {'NOMIS TS066':<25} {'21 March 2021':<30} {'2021 LSOA':<15} Census night")
print(f"  {'IMD 2025':<25} {'2025 (scores), mid-2022 (pop)':<30} {'2021 LSOA':<15} Pop denominator is mid-2022")
print(f"  {'RUC 2021':<25} {'2021':<30} {'2021 LSOA':<15} Aligned with Census")
print(f"  {'NaPTAN':<25} {'Live (downloaded today)':<30} {'N/A (coords)':<15} Snapshot — stops change daily")
print(f"  {'BODS GTFS':<25} {'Live (downloaded today)':<30} {'N/A (routes)':<15} Snapshot — timetables change")
print(f"  {'LSOA Boundaries':<25} {'Dec 2021':<30} {'2021 LSOA':<15} Same boundaries as all above")
print(f"  {'Region Boundaries':<25} {'Dec 2021':<30} {'2021 regions':<15} 9 England regions")
print()
print("⚠ KEY TEMPORAL MISMATCH:")
print("  IMD population denominator is mid-2022, not Census night 2021.")
print("  Mean per-LSOA difference: ~40 people (population growth in 15 months).")
print("  Impact: negligible for rates/percentages. Use Census 2021 population")
print("  for all per-capita metrics. Use IMD mid-2022 ONLY when specifically")
print("  working with IMD denominators.")

---
# Part 6: Derived Metrics Catalog

Every formula the pipeline will compute, locked with exact column names.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# DERIVED METRICS CATALOG
# ══════════════════════════════════════════════════════════════════════════

print("DERIVED METRICS — EXACT FORMULAS FOR PIPELINE")
print("="*60)
print()

metrics = [
    ("unemployment_rate",
     "NOMIS 'Unemployed' / NOMIS 'Econ active (excl FT students)' × 100",
     "Percentage of economically active people who are unemployed"),
    ("elderly_pct",
     "(TS007a age_65_69 + age_70_74 + age_75_79 + age_80_84 + age_85plus) / TS007a 'Age: Total' × 100",
     "Percentage of population aged 65 and over"),
    ("young_pct",
     "(TS007a age_0_4 + age_5_9 + age_10_14) / TS007a 'Age: Total' × 100",
     "Percentage of population aged 0-14"),
    ("working_age_pct",
     "100 - elderly_pct - young_pct",
     "Percentage of population aged 15-64 (residual)"),
    ("nocar_pct",
     "TS045 'No cars or vans' / TS045 'Total: All households' × 100",
     "Percentage of HOUSEHOLDS (not people) with no car"),
    ("nonwhite_pct",
     "(TS021 'Total' - TS021 'White') / TS021 'Total' × 100",
     "Percentage of population that is non-white"),
    ("stops_per_1k",
     "stop_count / TS001 population × 1000",
     "Bus stops per 1000 population (density metric)"),
    ("bus_need_score",
     "imd_score/max × 50 + (nocar>30%) × 25 + (elderly>25%) × 25",
     "Composite 0-100 score of bus transport need (experimental)"),
]

print(f"{'Metric':<22} {'Formula':<75} {'Description'}")
print(f"{'─'*150}")
for name, formula, desc in metrics:
    print(f"{name:<22} {formula:<75} {desc}")

print()
print("DIRECT-FROM-SOURCE METRICS (no computation needed):")
print(f"  imd_score           → IMD 'Index of Multiple Deprivation (IMD) Score'")
print(f"  imd_rank            → IMD 'Index of Multiple Deprivation (IMD) Rank'")
print(f"  imd_decile          → IMD 'Index of Multiple Deprivation (IMD) Decile'")
print(f"  income_score        → IMD 'Income Score (rate)'  [0-1]")
print(f"  employment_score    → IMD 'Employment Score (rate)'  [0-1]")
print(f"  health_score        → IMD 'Health Deprivation and Disability Score'  [z-score]")
print(f"  education_score     → IMD 'Education, Skills and Training Score'")
print(f"  crime_score         → IMD 'Crime Score'  [z-score]")
print(f"  barriers_score      → IMD 'Barriers to Housing and Services Score'")
print(f"  living_env_score    → IMD 'Living Environment Score'")
print(f"  idaci_score         → IMD 'IDACI Score (rate)'  [0-1]")
print(f"  idaopi_score        → IMD 'IDAOPI Score (rate)'  [0-1]")
print(f"  population          → TS001 'Residence type: Total; measures: Value'")
print(f"  ruc_code            → RUC 'RUC21CD'  [UN1/UF1/RSN1/RLN1/RSF1/RLF1]")
print(f"  urban_rural         → RUC 'Urban_rural_flag'  [Urban/Rural]")
print()
print("POPULATION DENOMINATOR RULE:")
print("  Always use Census 2021 TS001 'Residence type: Total' for per-capita metrics.")
print("  NEVER use IMD mid-2022 population for per-capita rates.")
print("  Total England population: 56,490,056 (sum of 33,755 LSOAs)")

---
# Part 7: Edge Cases & Data Traps — Consolidated

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EDGE CASES & DATA TRAPS — EVERYTHING THE PIPELINE MUST HANDLE
# ══════════════════════════════════════════════════════════════════════════

print("DATA TRAPS & EDGE CASES — CONSOLIDATED REFERENCE")
print("="*60)
print()

traps = [
    ("NaPTAN", "StopType filter",
     "Must filter to BCT/BCS/BCE. 18 stop types exist (rail, metro, ferry etc)."),
    ("NaPTAN", "Status filter",
     "Must filter to 'active'. 45,840 inactive + 1 pending."),
    ("NaPTAN", "England filter",
     "ATCO prefix 0-4 = England. 5xx=Wales, 6xx=Scotland. Bounding box includes both."),
    ("NaPTAN", "Null coordinates",
     "52,449 rows have null lat/lon. Must use Easting/Northing or drop."),
    ("NaPTAN", "Duplicate stop names",
     "CommonName is NOT unique. 'Bus Station' appears 1000+ times. Use ATCOCode."),
    ("BODS", "Route vs trip vs pattern",
     "1 route → many trips → many stop_times. Count routes, not trips."),
    ("BODS", "Non-bus routes",
     "route_type 3=bus. Also 0=tram, 1=metro, 2=rail, 4=ferry, 200=coach."),
    ("BODS", "Stop ID format",
     "stop_id in BODS = ATCOCode in NaPTAN. But 85,787 BODS stops not in NaPTAN (Wales/Scotland)."),
    ("Census", "England filter",
     "All Census files have 35,672 rows (E+W). Filter 'geography code' to E* for England."),
    ("Census", "Population base",
     "TS001 = ALL residents. TS066 age 16+. NEVER mix denominators."),
    ("Census", "Household vs person",
     "TS045 car ownership is per HOUSEHOLD. Do NOT divide by population."),
    ("IMD", "Decile direction",
     "Decile 1 = MOST deprived 10%. Rank 1 = single most deprived LSOA."),
    ("IMD", "Score interpretation",
     "Income/Employment are rates (0-1). Health/Crime are z-scores. NOT additive."),
    ("IMD", "Population year",
     "IMD uses mid-2022 population estimates. Census is March 2021. Use Census for rates."),
    ("IMD", "LAD boundaries",
     "LAD codes are 2024 boundaries. LSOA codes are 2021 boundaries. Different!"),
    ("RUC", "Classification basis",
     "Distance-based (proximity to towns), NOT density-based. RSN1 is 'Rural' despite being near cities."),
    ("Car ownership", "Dual signal",
     "High no-car = poverty OR wealthy urban (London). Must cross-reference with IMD."),
    ("Spatial join", "Coastal orphans",
     "2 of 274,719 stops don't fall in any LSOA polygon (piers). Use nearest-neighbour fallback."),
    ("Population", "Denominator",
     "ALWAYS use ONS Census total (~56.49M). NEVER use pipeline-filtered count."),
    ("Gender proxy", "No direct data",
     "Factor 7 has no gender-specific data. Use IDACI + barriers + young_pct as proxy. Flag limitation."),
]

print(f"{'#':<4} {'Source':<15} {'Trap':<25} {'Description'}")
print(f"{'─'*120}")
for i, (source, trap, desc) in enumerate(traps, 1):
    print(f"{i:<4} {source:<15} {trap:<25} {desc}")

print(f"\n  Total traps documented: {len(traps)}")
print(f"  Every one of these must be handled in the pipeline.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SAVE MASTER TABLE + SUMMARY
# ══════════════════════════════════════════════════════════════════════════

# Save master LSOA table for pipeline use
AUDIT_DIR = ROOT / "data" / "audit"
master_path = AUDIT_DIR / "master_lsoa_table.parquet"
master.to_parquet(master_path, index=False)
print(f"Master table saved: {master_path}")
print(f"  {len(master):,} rows × {len(master.columns)} columns")
print(f"  Columns: {list(master.columns)}")
print()
print(f"DATA UNDERSTANDING COMPLETE.")
print(f"")
print(f"What we now know:")
print(f"  ✓ Every column in every dataset — what it means, what unit, what range")
print(f"  ✓ Every derived metric — exact formula with source column names")
print(f"  ✓ How all 8 factors relate to each other (correlation matrix)")
print(f"  ✓ How bus coverage relates to socio-economic need (the core finding)")
print(f"  ✓ Pattern-breaking LSOAs — why anomalies exist")
print(f"  ✓ Dataset join graph and temporal alignment")
print(f"  ✓ 20 data traps the pipeline must handle")
print(f"")
print(f"Next step: build src/aequitas/ pipeline with this understanding as the foundation.")